In [0]:
# COMMAND ----------
# CELL 1: Setup & Parameters
import re
import json
import traceback
import urllib.request
import urllib.error
from pyspark.sql.functions import col, lit, current_timestamp
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, TimestampType, DoubleType, BooleanType
)
from sodapy import Socrata

# Tag all Spark SQL queries for tracking
spark.conf.set('spark.databricks.queryTag.data_origin', 'open_data_portal')


# -------------------------------------------------------------------------
# State Container (Tracked globally to ensure audit logging always has context)
# -------------------------------------------------------------------------
EXEC_STATE = {
    "dataset_id": "UNKNOWN",
    "title": "UNKNOWN",
    "table_name": "UNKNOWN",
    "category": "Uncategorized",
    "portal_data_update": None,
    "records_ingested": 0
}

# -------------------------------------------------------------------------
# Helper Functions
# -------------------------------------------------------------------------
def log(*args, **kwargs):
    if VERBOSE:
        print(*args, **kwargs)

def sanitize_table_name(name: str) -> str:
    clean = re.sub(r'[^a-zA-Z0-9_]', '_', (name or '').lower())
    clean = re.sub(r'_+', '_', clean).strip('_')
    return clean[:200]

def sql_str(val) -> str:
    if val is None:
        return ""
    return str(val).strip().replace("\\", "\\\\").replace("'", "\\'")

def append_log_event(
    dataset_id: str, 
    title: str, 
    table_name: str, 
    category: str, 
    portal_data_update: str, 
    status: str, 
    records: int = 0, 
    error: str = ""
):
    """
    Appends an immutable audit row to the queue table.
    Enclosed in a strict try-except to prevent logging failures from masking primary errors.
    """
    try:
        schema = StructType([
            StructField("dataset_id", StringType(), False),
            StructField("task_type", StringType(), True),
            StructField("title", StringType(), True),
            StructField("table_name", StringType(), True),
            StructField("category", StringType(), True),
            StructField("portal_data_updated_at", StringType(), True),
            StructField("portal_metadata_updated_at", StringType(), True),
            StructField("status", StringType(), True),
            StructField("records_ingested", LongType(), True),
            StructField("error_message", StringType(), True)
        ])

        resolved_data_update = portal_data_update  or None

        # Clean, truncate, and extract the most relevant error info
        formatted_error = None
        if error:
            formatted_error = str(error).strip()[:1000]

        log_row = [{
            "dataset_id": str(dataset_id or EXEC_STATE["dataset_id"]),
            "task_type": "DATA",
            "title": str(title or EXEC_STATE["title"]),
            "table_name": str(table_name or EXEC_STATE["table_name"]),
            "category": str(category or EXEC_STATE["category"]),
            "portal_data_updated_at": resolved_data_update,
            "portal_metadata_updated_at": None,
            "status": status,
            "records_ingested": int(records or 0),
            "error_message": formatted_error
        }]

        (spark.createDataFrame(log_row, schema=schema)
            .withColumn("logged_at", current_timestamp())
            .write
            .format("delta")
            .mode("append")
            .option("mergeSchema", "true")
            .saveAsTable(QUEUE_TABLE))
        
        log(f"[AUDIT] Successfully logged status '{status}' for dataset '{dataset_id}'.")
    except Exception as log_failure:
        # Fallback to driver log so logging issues are visible in cluster stderr
        print(f"[FATAL AUDIT ERROR] Could not persist audit log to Delta table: {log_failure}")
        print(f"[ORIGINAL CONTEXT] dataset_id={dataset_id}, status={status}, error={error}")

# -------------------------------------------------------------------------
# Cell 1 Execution Wrapped in Top-Level Guard
# -------------------------------------------------------------------------
try:
    dbutils.widgets.text("domain", "opendata.maryland.gov", "Socrata Domain")
    dbutils.widgets.text("payload", "{}", "JSON Task Payload")
    dbutils.widgets.text("catalog_name", "open_data_portal", "Catalog Name")
    dbutils.widgets.text("schema_name", "bronze", "Target Schema Name")
    dbutils.widgets.text("meta_schema", "meta", "Control Schema Name")
    dbutils.widgets.dropdown("verbose", "true", ["true", "false"], "Verbose Logging")

    SOCRATA_DOMAIN = dbutils.widgets.get("domain")
    CATALOG_NAME = dbutils.widgets.get("catalog_name")
    SCHEMA_NAME = dbutils.widgets.get("schema_name")
    META_SCHEMA = dbutils.widgets.get("meta_schema")
    VERBOSE = dbutils.widgets.get("verbose").lower() == "true"

    QUEUE_TABLE = f"`{CATALOG_NAME}`.`{META_SCHEMA}`.`_socrata_ingestion_queue`"

    # Unpack Payload
    raw_payload = dbutils.widgets.get("payload")
    if raw_payload and raw_payload != "{}":
        task_data = json.loads(raw_payload)
        DATASET_ID = task_data.get("id", "").strip()
        PORTAL_DATA_UPDATED_AT = task_data.get("up", "").strip()
    else:
        DATASET_ID = dbutils.widgets.get("dataset_id").strip()
        PORTAL_DATA_UPDATED_AT = dbutils.widgets.get("portal_data_updated_at").strip()

    if not DATASET_ID:
        raise ValueError("dataset_id widget must be provided for single-task ingestion.")

    # Populate state tracker
    EXEC_STATE["dataset_id"] = DATASET_ID
    EXEC_STATE["portal_data_update"] = PORTAL_DATA_UPDATED_AT or None

except BaseException as setup_err:
    # Catches JSON errors, missing widgets, or initialization errors
    append_log_event(
        dataset_id=EXEC_STATE["dataset_id"],
        title="Setup Initialization",
        table_name="N/A",
        category="Setup",
        portal_data_update=EXEC_STATE["portal_data_update"],
        status="FAILED",
        records=0,
        error=f"Cell 1 Setup Failure: {setup_err}\n{traceback.format_exc()}"
    )
    raise RuntimeError(f"Initialization failed: {setup_err}") from setup_err

In [0]:
# COMMAND ----------
# CELL 2: Ingest, Transform, and Save Single Unified Table
import os
import uuid
import shutil
import traceback
from requests.exceptions import RequestException, HTTPError
import pyarrow as pa
import pyarrow.parquet as pq
from tenacity import (
    retry,
    stop_after_attempt,
    wait_exponential,
    retry_if_exception
)

STAGING_DIR = None

def is_retryable_error(exception: BaseException) -> bool:
    if isinstance(exception, urllib.error.HTTPError):
        return exception.code in (429, 500, 502, 503, 504)
    if isinstance(exception, HTTPError) and exception.response is not None:
        return exception.response.status_code in (429, 500, 502, 503, 504)
    if isinstance(exception, (RequestException, urllib.error.URLError, TimeoutError, ConnectionError)):
        return True
    return False

soda_retry = retry(
    reraise=True,
    stop=stop_after_attempt(5),
    wait=wait_exponential(multiplier=2, min=2, max=60),
    retry=retry_if_exception(is_retryable_error),
    before_sleep=lambda rs: log(
        f"    [RETRY] Attempt {rs.attempt_number} failed ({rs.outcome.exception()}). "
        f"Retrying in {rs.next_action.sleep:.2f}s..."
    )
)

@soda_retry
def fetch_metadata_with_retry(soda_client, d_id):
    return soda_client.get_metadata(d_id)

@soda_retry
def fetch_page_with_retry(soda_client, d_id, limit, offset, order=":id"):
    try:
        return soda_client.get(d_id, limit=limit, offset=offset, order=order)
    except Exception:
        return soda_client.get(d_id, limit=limit, offset=offset)

GEO_TYPES = {
    'point', 'multipoint', 'linestring', 'multilinestring', 
    'polygon', 'multipolygon', 'location', 'geometry'
}

# -------------------------------------------------------------------------
# Top-Level Execution Guard
# -------------------------------------------------------------------------
try:
    SECRET_SCOPE = "socrata_scope"
    try:
        APP_TOKEN = dbutils.secrets.get(scope=SECRET_SCOPE, key="app_token")
        log("Loaded Socrata app token from secret scope.")
    except Exception:
        APP_TOKEN = None
        log("Warning: Running unauthenticated.")

    client = Socrata(SOCRATA_DOMAIN, APP_TOKEN, timeout=600)
    log(f"\nProcessing Dataset: {DATASET_ID}")

    # 1. Fetch Upstream Metadata & Validate Public Status
    try:
        dataset_metadata = fetch_metadata_with_retry(client, DATASET_ID)
    except urllib.error.HTTPError as http_err:
        if http_err.code in (403, 404):
            append_log_event(
                dataset_id=DATASET_ID, 
                title=DATASET_ID, 
                table_name=DATASET_ID, 
                category="Uncategorized", 
                portal_data_update=None, 
                status="DEPRECATED", 
                records=0, 
                error=f"HTTP {http_err.code}"
            )
            dbutils.notebook.exit(f"DEPRECATED_HTTP_{http_err.code}")
        raise http_err

    dataset_title = dataset_metadata.get('name', DATASET_ID)
    category = dataset_metadata.get("category") or "Uncategorized"
    table_name = sanitize_table_name(dataset_title)
    target_table_path = f"`{CATALOG_NAME}`.`{SCHEMA_NAME}`.`{table_name}`"

    # Update Global State for logging continuity
    EXEC_STATE.update({
        "title": dataset_title,
        "table_name": table_name,
        "category": category,
        "portal_data_update":  PORTAL_DATA_UPDATED_AT
    })

    approvals = (dataset_metadata.get("approvals") or [{}])[0] or {}
    target_audience = (approvals.get("targetAudience") or "").lower()
    publication_stage = (dataset_metadata.get("publicationStage") or "").lower()

    if not (publication_stage == "published" and target_audience == "public"):
        append_log_event(
            DATASET_ID, dataset_title, table_name, category, 
            EXEC_STATE["portal_data_update"], "DEPRECATED", 0, "Not public/published"
        )
        dbutils.notebook.exit("Filtered non-public dataset.")

    # 2. Schema Discovery from Socrata Column Definitions
    socrata_columns = dataset_metadata.get('columns', [])
    all_meta_columns = [c.get('fieldName') for c in socrata_columns if c.get('fieldName')]

    geo_cols = set()
    socrata_type_cast_map = {}

    for c in socrata_columns:
        fn = c.get('fieldName')
        dt = (c.get('dataTypeName') or '').lower()
        if not fn:
            continue
        if dt in GEO_TYPES:
            geo_cols.add(fn)
        elif dt in ['number', 'money']:
            socrata_type_cast_map[fn] = 'double'
        elif dt == 'checkbox':
            socrata_type_cast_map[fn] = 'boolean'

    serverless_input_schema = StructType([
        StructField(col_name, StringType(), True) for col_name in all_meta_columns
    ])
    staging_arrow_schema = pa.schema(
        [pa.field(col_name, pa.string()) for col_name in all_meta_columns]
    )
    cast_exprs = {c: col(c).cast(t) for c, t in socrata_type_cast_map.items()}

    # 3. Stream SODA Rows to Disk Staging
    STAGING_DIR = f"/Volumes/{CATALOG_NAME}/{META_SCHEMA}/socrata_staging/{DATASET_ID}_{uuid.uuid4().hex[:8]}"
    os.makedirs(STAGING_DIR, exist_ok=True)

    offset = 0
    BATCH_SIZE = 50000 if len(all_meta_columns) <= 25 else 20000
    total_records = 0
    batch_index = 0

    while True:
        records = fetch_page_with_retry(client, DATASET_ID, limit=BATCH_SIZE, offset=offset, order=":id")
        if not records:
            break
        
        chunk_path = os.path.join(STAGING_DIR, f"part_{batch_index:05d}.parquet")

        batch_rows = []
        for r in records:
            row_dict = {}
            for col_name in all_meta_columns:
                val = r.get(col_name)
                if val is None:
                    row_dict[col_name] = None
                elif isinstance(val, (dict, list)):
                    row_dict[col_name] = json.dumps(val)
                else:
                    val_str = str(val).strip()
                    if col_name in socrata_type_cast_map and socrata_type_cast_map[col_name] == 'double':
                        val_str = re.sub(r'[\$,]', '', val_str)
                    row_dict[col_name] = val_str if val_str else None
            batch_rows.append(row_dict)

        pq.write_table(
            pa.Table.from_pylist(batch_rows, schema=staging_arrow_schema),
            chunk_path
        )

        total_records += len(records)
        EXEC_STATE["records_ingested"] = total_records
        batch_index += 1
        offset += BATCH_SIZE
        
        log(f"    Staged batch {batch_index}: {len(records)} rows written (Total: {total_records}).")
        
        if len(records) < BATCH_SIZE:
            break

    if total_records == 0:
        append_log_event(
            DATASET_ID, dataset_title, table_name, category, 
            EXEC_STATE["portal_data_update"], "EMPTY", 0, None
        )
        dbutils.notebook.exit("EMPTY")

    # 4. Read Distributed Dataset via Spark 
    df = (
        spark.read
        .schema(serverless_input_schema)
        .parquet(STAGING_DIR)
    )
    if cast_exprs:
        df = df.withColumns(cast_exprs)

    loc_col = next((c for c in geo_cols if c in df.columns), None)
    if not loc_col:
        lower_cols = {c.lower(): c for c in df.columns}
        loc_col = next((lower_cols[c] for c in ["location", "location_1", "the_geom", "geom", "geometry"] if c in lower_cols), None)

    sample_val = ""
    if loc_col:
        sample_row = df.select(loc_col).filter(col(loc_col).isNotNull() & (F.trim(col(loc_col)) != "")).limit(1).collect()
        sample_val = str(sample_row[0][loc_col]).strip() if sample_row else ""

    has_geometry = False

    if loc_col and ("human_address" in sample_val or "latitude" in sample_val):
        df = (
            df.withColumn("street_address", F.col(loc_col).cast("string"))
              .withColumn("street_address", F.get_json_object(F.get_json_object(col(loc_col), "$.human_address"), "$.address"))
              .withColumn("city", F.get_json_object(F.get_json_object(col(loc_col), "$.human_address"), "$.city"))
              .withColumn("state", F.get_json_object(F.get_json_object(col(loc_col), "$.human_address"), "$.state"))
              .withColumn("zip", F.expr(f"cast(get_json_object(get_json_object({loc_col}, '$.human_address'), '$.zip') AS STRING)"))
              .withColumn("__lat", F.expr(f"try_cast(get_json_object({loc_col}, '$.latitude') AS DOUBLE)"))
              .withColumn("__lon", F.expr(f"try_cast(get_json_object({loc_col}, '$.longitude') AS DOUBLE)"))
        )
        has_coords = df.filter(col("__lat").isNotNull() & col("__lon").isNotNull()).limit(1).count() > 0
        if has_coords:
            has_geometry = True
            df = df.withColumn(
                "__g",
                F.expr("""
                    CASE 
                        WHEN __lon BETWEEN -180 AND 180 AND __lat BETWEEN -90 AND 90 AND NOT (__lon = 0 AND __lat = 0)
                        THEN st_point(__lon, __lat, 4326)
                        ELSE NULL 
                    END
                """)
            )

    elif "latitude" in [c.lower() for c in df.columns] and "longitude" in [c.lower() for c in df.columns]:
        l_map = {c.lower(): c for c in df.columns}
        lat_c, lon_c = l_map["latitude"], l_map["longitude"]
        df = (
            df.withColumn("__lat", F.expr(f"try_cast({lat_c} AS DOUBLE)"))
              .withColumn("__lon", F.expr(f"try_cast({lon_c} AS DOUBLE)"))
        )
        has_coords = df.filter(col("__lat").isNotNull() & col("__lon").isNotNull()).limit(1).count() > 0
        if has_coords:
            has_geometry = True
            df = df.withColumn(
                "__g",
                F.expr("""
                    CASE 
                        WHEN __lon BETWEEN -180 AND 180 AND __lat BETWEEN -90 AND 90 AND NOT (__lon = 0 AND __lat = 0)
                        THEN st_point(__lon, __lat, 4326)
                        ELSE NULL 
                    END
                """)
            )

    elif loc_col and '"coordinates"' in sample_val:
        has_geometry = True
        df = df.withColumn(
            "__g",
            F.expr(f"CASE WHEN {loc_col} IS NOT NULL AND length(trim({loc_col})) > 0 THEN st_setsrid(st_geomfromgeojson({loc_col}), 4326) ELSE NULL END")
        )

    for z_candidate in ["zip", "zip_code", "zipcode", "postal_code"]:
        if z_candidate in df.columns:
            df = df.withColumn(z_candidate, col(z_candidate).cast("string"))

    if has_geometry:
        df = (
            df.withColumn("geom", F.expr("__g"))
              .withColumn("geom_md", F.expr("st_transform(__g, 6487)"))
              .withColumn(
                  "h3_r8",
                  F.expr("""
                      CASE 
                          WHEN geom IS NULL OR st_isempty(geom) THEN NULL
                          WHEN abs(st_x(st_centroid(geom))) > 180 OR abs(st_y(st_centroid(geom))) > 90 THEN NULL
                          ELSE h3_longlatash3(st_x(st_centroid(geom)), st_y(st_centroid(geom)), 8)
                      END
                  """)
              )
        )

    drop_candidates = ["__g", "__lat", "__lon"]
    if loc_col:
        drop_candidates.append(loc_col)

    cols_to_drop = [
        c for c in df.columns 
        if c in drop_candidates 
        or c.startswith("@computed_region") 
        or c.startswith(":")
    ]
    df = df.drop(*cols_to_drop)

    # 5. Commit to Delta Lake
    clean_target = target_table_path.replace("`", "")
    if spark.catalog.tableExists(clean_target):
        if not (has_geometry and "h3_r8" in df.columns):
            try:
                spark.sql(f"ALTER TABLE {target_table_path} UNSET TBLPROPERTIES IF EXISTS ('delta.dataSkippingStatsColumns')")
            except Exception as e:
                log(f"Pre-write property cleanup warning: {e}")

    (
        df.write
          .format("delta")
          .mode("overwrite")
          .option("overwriteSchema", "true")
          .saveAsTable(target_table_path)
    )
    log(f"Committed unified table: {target_table_path} ({total_records} rows).")

    # 5b. Update Table Metadata
    resolved_data_ts = EXEC_STATE["portal_data_update"]
    if resolved_data_ts:
        clean_ts = sql_str(str(resolved_data_ts))
        try:
            spark.sql(f"""
                ALTER TABLE {target_table_path} 
                SET TBLPROPERTIES ('open_data.data_updated_at' = '{clean_ts}')
            """)
        except Exception as prop_err:
            log(f"Warning: Failed to set table property: {prop_err}")

    # 6. Log Success
    append_log_event(
        dataset_id=DATASET_ID,
        title=EXEC_STATE["title"],
        table_name=EXEC_STATE["table_name"],
        category=EXEC_STATE["category"],
        portal_data_update=EXEC_STATE["portal_data_update"],
        status="SUCCESS",
        records=total_records,
        error=None
    )

except BaseException as pipeline_err:
    # 1. Only allow genuine dbutils.notebook.exit() calls to bubble up untouched
    err_str = str(pipeline_err)
    if "NotebookExit" in type(pipeline_err).__name__ or "dbutils.notebook.exit" in err_str:
        raise pipeline_err

    stack_trace = traceback.format_exc()
    error_summary = f"{err_str}\n{stack_trace}"

    # 2. Check historical consecutive failures for this dataset
    MAX_CONSECUTIVE_FAILURES = 3
    failure_status = "FAILED"

    try:
        failure_history_df = spark.sql(f"""
            WITH ranked_events AS (
                SELECT 
                    status,
                    ROW_NUMBER() OVER (ORDER BY logged_at DESC) AS rn
                FROM {QUEUE_TABLE}
                WHERE dataset_id = '{EXEC_STATE["dataset_id"]}'
                  AND task_type = 'DATA'
            ),
            latest_run AS (
                SELECT coalesce(min(rn) - 1, (SELECT count(1) FROM ranked_events)) AS streak
                FROM ranked_events
                WHERE status NOT IN ('FAILED')
            )
            SELECT streak FROM latest_run
        """).collect()

        recent_failures = failure_history_df[0]["streak"] if failure_history_df else 0
        if (recent_failures + 1) >= MAX_CONSECUTIVE_FAILURES:
            failure_status = "QUARANTINED"
            log(f"[CIRCUIT BREAKER] Dataset {EXEC_STATE['dataset_id']} exceeded failure limit ({recent_failures + 1} times). Setting status to QUARANTINED.")
    except Exception as count_err:
        log(f"Warning: Could not compute failure streak: {count_err}")

    # 3. Log either FAILED or QUARANTINED to _socrata_ingestion_queue
    append_log_event(
        dataset_id=EXEC_STATE["dataset_id"],
        title=EXEC_STATE["title"],
        table_name=EXEC_STATE["table_name"],
        category=EXEC_STATE["category"],
        portal_data_update=EXEC_STATE["portal_data_update"],
        status=failure_status,
        records=EXEC_STATE["records_ingested"],
        error=error_summary
    )

    # 4. Gracefully exit so sibling for_each iterations are unaffected
    dbutils.notebook.exit(f"{failure_status}: {err_str}")

finally:
    # Cleanup disk staging directory regardless of success or failure
    if STAGING_DIR and os.path.exists(STAGING_DIR):
        try:
            shutil.rmtree(STAGING_DIR)
            log(f"Cleaned up staging folder: {STAGING_DIR}")
        except Exception as cleanup_err:
            log(f"Warning: Failed to clean up staging folder: {cleanup_err}")